### Task 1: Build the Core LCEL Chain with a ChatPromptTemplate

Create a new Jupyter notebook called smart_qa_bot.ipynb. Import ChatOpenAI and build a ChatPromptTemplate with a system message that defines your bot as a helpful assistant, and a human message placeholder for {question}. Use the LCEL pipe operator (|) to chain: prompt | llm | StrOutputParser(). Invoke the chain with a sample question and print the response. Verify the chain runs without errors before moving to the next task.

In [6]:
from langchain_core.prompts import ChatPromptTemplate

from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

# get ANTHROPIC_API_KEY env var from .env file
load_dotenv()


def core_lcel():
    model = init_chat_model(
            model="claude-sonnet-4-5-20250929",
            model_provider="anthropic",
            temperature=0.7,
            streaming=False,
            max_retries=3,
        )
    
    prompt = ChatPromptTemplate.from_messages([
                ("system", "You are a helpful assistant."),
                ("human", "{question}")
            ])

    parser = StrOutputParser()

    chain = prompt | model | parser

    response = chain.invoke({"question": "What is the capital of Italy?"})

    print(f"Response: {response}")

core_lcel()

Response: The capital of Italy is Rome.


### Task 2: Add Structured Output with a Pydantic Model

Define a Pydantic BaseModel called QAResponse with three fields: answer (str), confidence (float between 0 and 1), and sources (List[str]). Replace StrOutputParser() in your chain with llm.with_structured_output(QAResponse). Invoke the chain with a question and print the structured result, accessing each field individually (e.g., result.answer, result.confidence). Confirm the output is a typed Python object, not a plain string.

In [15]:
from pydantic import BaseModel, Field
from typing import List

load_dotenv()

class QAResponse(BaseModel):
    answer: str = Field(description="ai answer for the questions")
    confidence: float = Field(description="confidence level between 0 and 1", ge=0.0, le=1.0)
    sources: List[str] = Field(description="sources", default_factory=list)
    

def add_structured_output_with_pydeantic_model():
    model = init_chat_model(
            model="claude-sonnet-4-5-20250929",
            model_provider="anthropic",
            temperature=0.7,
            streaming=False,
            max_retries=3,
        ).with_structured_output(QAResponse)
    

    prompt = ChatPromptTemplate.from_messages([
                    ("system", "You are a helpful assistant."),
                    ("human", "{question}")
                ])
    
    chain = prompt | model

    result = chain.invoke({"question": "How old is the Earth?"})

    print(type(result))
    print(f"""
Answer: {result.answer}, 
Confidence: {result.confidence}, 
Sources: {result.sources}""")


add_structured_output_with_pydeantic_model()


<class '__main__.QAResponse'>

Answer: The Earth is approximately 4.54 billion years old. This age has been determined through radiometric dating of rocks and meteorites, particularly through the analysis of the oldest known terrestrial rocks and moon rocks, as well as meteorites that formed at the same time as our solar system. Scientists use various dating methods, with uranium-lead dating of zircon crystals being one of the most reliable techniques. The age of 4.54 billion years (with an uncertainty of about 50 million years) represents the time since Earth formed from the solar nebula along with the rest of the solar system., 
Confidence: 0.99, 
Sources: ['Scientific consensus based on radiometric dating methods', 'Geological and planetary science research']


### Task 3: Configure Multi-Provider Fallback with OpenAI and Anthropic

Instantiate both ChatOpenAI (gpt-4o-mini) and ChatAnthopic (claude-3-haiku-20240307). Use the .with_fallbacks() method to create a resilient primary_llm that falls back to Anthropic if OpenAI fails. Build a new chain: prompt | primary_llm | StrOutputParser(). Test the fallback by first running a normal invocation, then simulate a failure by passing an invalid OpenAI API key to a separate ChatOpenAI instance and confirm the chain automatically uses Anthropic. Print which model produced each response.

In [30]:
# from langchain_anthropic import ChatAnthropic
#from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv

load_dotenv()

# llm_openai = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
# llm_openai.with_fallbacks()
# llm_anthropic = ChatAnthropic(model="claude-sonnet-4-5-20250929", temperature=0)


primary_llm = init_chat_model(
    model="gpt-4o-mini",
    model_provider="openai",
    temperature=0.3,
    streaming=False,
    max_retries=3,
)


fallback_llm = init_chat_model(
    model="claude-sonnet-4-5-20250929",
    model_provider="anthropic",
    temperature=0.7,
    streaming=False,
    max_retries=3,
)

model = primary_llm.with_fallbacks([fallback_llm])

parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

chain = prompt | model

response = chain.invoke({"question": "Tell me a joke."})

print(f"Model: {response.response_metadata.get("model_name")}")

result = parser.invoke(response)
print(f"Answer: {result}")




Model: claude-sonnet-4-5-20250929
Answer: Why don't scientists trust atoms?

Because they make up everything!


### Task 4: Implement Real-Time Streaming and Batch Execution

- Part A - Streaming: Use chain.stream({"question": "Explain LangChain in simple terms"}) and iterate over the chunks, printing each token as it arrives (simulating real-time output). 
- Part B - Batch: Create a list of 5 different questions and pass them all at once using chain.batch([...], config={"max_concurrency": 3}). 
- Print all 5 responses. Observe the time difference between batch and sequential invocations by timing both with Python's time module.

In [42]:
# Part A
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
import time

load_dotenv()

model = init_chat_model(
    model="claude-sonnet-4-5-20250929",
    model_provider="anthropic",
    temperature=0.7,
    streaming=True,
    max_retries=3,
)

prompt = ChatPromptTemplate.from_template("Answer the {question} as a helpful asistant.")

parser = StrOutputParser()

chain = prompt | model | parser

print("Streaming: \n")

start = time.perf_counter()

for chunk in chain.stream({"question": "Explain LangChain in simple terms"}):
    print(f"[{chunk}]", end="", flush=True)

end = time.perf_counter()

print("\n\nStreaming time: ", end - start, "seconds")


Streaming: 

[][#][ LangChain Explained Simply

**LangChain is a framework that helps developers build applications powered by large language models (][LLMs) like ChatGPT.**

Think of it as a **toolkit that makes AI][ apps easier to create**. Here's what it does:

## Key Features:][

### 1. **Chains**
- Links multiple steps together
- Example: Ask a question → Search][ for info → Generate an answer
- Like building blocks that connect AI tasks][

### 2. **Memory**
- Helps AI remember previous conversations
- Makes chatbots feel more natural and][ contextual

### 3. **Data Connection**
- Lets AI access your own documents][, databases, or websites
- Example: Chat with your PDF files or company][ data

### 4. **Agents**
- AI that can use tools and make decisions
- Can][ search the web, do calculations, or access APIs automatically

## Simple Analogy:][
If an LLM (like ChatGPT) is a smart brain, **][LangChain is the body and tools** that let that brain interact with the real][ world - readi

In [45]:
# Part B
import time
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

questions = [
    {"question": "What is LangChain?"},
    {"question": "What is an LLM?"},
    {"question": "What is LCEL?"},
    {"question": "What is a prompt template?"},
    {"question": "What is a chain in LangChain?"},
]

model = init_chat_model(
    model="claude-sonnet-4-5-20250929",
    model_provider="anthropic",
    temperature=0.7,
    streaming=False,
    max_retries=3,
)

parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template("You are a helpful asistant. Answer the {question}")

chain = prompt | model | parser

start = time.perf_counter()

results = chain.batch(questions, config={"max_concurrency": 3})

for result in zip(questions, results):
    print(f"\n\nquestion: {result[0]['question']} \n=> answer: {result[1]}")

end = time.perf_counter()

print("\n\nBatch time:", end - start, "seconds")



question: What is LangChain? 
=> answer: # What is LangChain?

**LangChain** is an open-source framework designed to simplify the development of applications powered by large language models (LLMs). It provides tools and abstractions that make it easier to build complex, production-ready AI applications.

## Key Features:

1. **Modular Components** - Pre-built modules for common LLM tasks like prompts, chains, and agents

2. **Chain Creation** - Connect multiple LLM calls and operations together in sequences

3. **Memory Management** - Add context and conversation history to your applications

4. **Data Integration** - Connect LLMs to external data sources and APIs

5. **Agent Systems** - Build autonomous agents that can use tools and make decisions

## Common Use Cases:

- **Chatbots** - Conversational AI with memory and context
- **Document Q&A** - Query and analyze documents using natural language
- **Data Analysis** - Extract insights from structured/unstructured data
- **Automat

### Task 5: Assemble the Complete Smart Q&A Bot with a CLI Loop

Combine everything from Tasks 1-4 into a single final smart_qa_bot.py script. The bot should: 
- (1) use a ChatPromptTemplate with a system message and human message, 
- (2) use the fallback chain (OpenAI with Anthropic backup), 
- (3) return a structured QAResponse Pydantic object, and 
- (4) stream the answer token-by-token to the terminal. 
- Wrap the chain in a while True loop that reads user input, exits on 'quit', and streams responses live. 
- Run at least 3 questions through the bot to confirm it works end-to-end.

In [46]:
from dotenv import load_dotenv
from typing import List

from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()


class QAResponse(BaseModel):
    answer: str = Field(description="Answer to the user's question")
    confidence: float = Field(
        description="Confidence level between 0 and 1",
        ge=0.0,
        le=1.0,
    )
    sources: List[str] = Field(
        description="Sources used",
        default_factory=list,
    )


# Primary model (OpenAI)
primary_llm = init_chat_model(
    model="gpt-4o-mini",
    model_provider="openai",
    temperature=0.3,
    streaming=True,
    max_retries=3,
)

# Fallback model (Anthropic)
fallback_llm = init_chat_model(
    model="claude-sonnet-4-5-20250929",
    model_provider="anthropic",
    temperature=0.7,
    streaming=True,
    max_retries=3,
)

# Shared fallback model
fallback_model = primary_llm.with_fallbacks([fallback_llm])


prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. "
            "Answer clearly and concisely."
        ),
        (
            "human",
            "{question}"
        ),
    ]
)

# Chain used for streaming text
stream_chain = (
    prompt
    | fallback_model
    | StrOutputParser()
)

# Chain used for structured output
structured_chain = (
    prompt
    | fallback_model.with_structured_output(QAResponse)
)


while True:

    question = input("\nYou: ")

    if question.lower() == "quit":
        print("Goodbye!")
        break

    print("\nAssistant: ", end="", flush=True)

    # Stream answer
    for token in stream_chain.stream(
        {"question": question}
    ):
        print(token, end="", flush=True)

    print()

    # Structured response
    qa = structured_chain.invoke(
        {"question": question}
    )

    print("\nStructured Response")
    print("-------------------")
    print(f"Confidence: {qa.confidence:.2f}")
    print(f"Sources: {qa.sources}")


Assistant: I'm Claude, an AI assistant made by Anthropic.

Structured Response
-------------------
Confidence: 1.00
Sources: []

Assistant: I don't have an age in the traditional sense. I'm Claude, an AI assistant made by Anthropic. I don't experience time the way humans do, and I don't have a creation date that I'm aware of or that defines my "age." Each conversation with me is independent, without memory of previous interactions.

Structured Response
-------------------
Confidence: 0.95
Sources: []
Goodbye!
